# Verify the small sensor-drift fixture

Read the configuration and completed native Spark results. This notebook does not regenerate the source or retrain a model. Both scenarios retain the same failures while sensor measurements change according to the declared configuration.

In [ ]:
"""Read the exact sensor simulation configuration used by the native Spark verification."""
from pathlib import Path
import json, os, hashlib, html
from IPython.display import display, HTML
configuration_path = Path("generator/configs/sensor-drift-small.json")
configuration = json.loads(configuration_path.read_text())
configuration

In [ ]:
"""Read sealed measurements and show source/label/readback checks without rerunning Spark."""
report_path = Path(os.environ["IOT_SENSOR_DRIFT_REPORT"])
report = json.loads(report_path.read_text())
assert report["passed"] and report["config"] == configuration
{"report_sha256": hashlib.sha256(report_path.read_bytes()).hexdigest(), "unique_source_rows_per_scenario": report["unique_telemetry_rows_per_scenario"], "joined_rows_per_scenario": report["joined_rows_per_scenario"], "labels_unchanged": report["labels_unchanged"], "all_persisted_values_readback_equal": report["all_persisted_values_readback_equal"], "sensor_effect_check": report["sensor_effect_check"]}

In [ ]:
"""Display genuine merged id/label/feature rows collected by the verified Spark job."""
columns = ["id", "label", "machine_id", "event_timestamp", "temperature_mean_10m", "vibration_max_10m"]
for scenario in ["baseline", "drift"]:
    rows = report["samples"][scenario][:4]
    header = "".join("<th>" + html.escape(name) + "</th>" for name in columns)
    body = ''.join('<tr>' + ''.join('<td>' + html.escape(str(row[name])) + '</td>' for name in columns) + '</tr>' for row in rows)
    display(HTML("<h3>" + scenario + "</h3><table><thead><tr>" + header + "</tr></thead><tbody>" + body + "</tbody></table>"))
assert [row["id"] for row in report["samples"]["baseline"]] == [row["id"] for row in report["samples"]["drift"]]
assert [row["label"] for row in report["samples"]["baseline"]] == [row["label"] for row in report["samples"]["drift"]]